# Un agente simple con un LLM local

Vamos a construir, paso a paso, un agente que usa un modelo corriendo localmente con [Ollama](https://ollama.com) y tres herramientas muy simples: una calculadora, un reloj y un lector de archivos.

**Antes de correr este notebook:**
1. Instala Ollama y déjalo corriendo (`ollama serve`).
2. Descarga un modelo con soporte de *tool-calling*: `ollama pull qwen2.5` (o `llama3.1`).
3. Instala el cliente de Python: `pip install ollama`.

No usaremos ningún framework de agentes — la idea es entender el ciclo completo con código explícito.

## Paso 0: imports y configuración

Importamos el cliente de `ollama` y fijamos el nombre del modelo que vamos a usar.

In [ ]:
import ollama
import datetime
import os

MODELO = "qwen2.5"  # cambia esto si descargaste otro modelo

## Paso 1: las herramientas

Son funciones Python normales y corrientes. El agente solo podrá hacer aquello para lo que le demos una herramienta — no tiene acceso a nada más que esto.

In [ ]:
def calculadora(expresion: str) -> str:
    """Evalua una expresion aritmetica simple, ej. '3 * (4 + 2)'."""
    permitidos = set("0123456789+-*/(). ")
    if not set(expresion) <= permitidos:
        return "Error: la expresion contiene caracteres no permitidos."
    try:
        return str(eval(expresion, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error al evaluar: {e}"


def fecha_actual() -> str:
    """Devuelve la fecha y hora actual del sistema."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def leer_archivo(ruta: str) -> str:
    """Lee y devuelve el contenido de un archivo de texto local."""
    if not os.path.isfile(ruta):
        return f"Error: no existe el archivo '{ruta}'."
    with open(ruta, "r", encoding="utf-8") as f:
        return f.read()[:2000]


# Probemos que funcionan de forma normal, sin ningun LLM de por medio:
print(calculadora("6 * 7"))
print(fecha_actual())

## Paso 2: describir las herramientas al modelo (tool schemas)

El LLM no puede "ver" el código Python de arriba. Necesitamos describirle, en un formato estructurado (JSON), qué herramientas existen, qué hacen y qué argumentos esperan. Con eso, el modelo puede *pedir* que se llame a una herramienta — pero no la ejecuta él mismo.

In [ ]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "calculadora",
            "description": "Evalua una expresion aritmetica y devuelve el resultado.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expresion": {
                        "type": "string",
                        "description": "Expresion aritmetica, ej. '12 * 7'",
                    }
                },
                "required": ["expresion"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "fecha_actual",
            "description": "Devuelve la fecha y hora actual del sistema.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "leer_archivo",
            "description": "Lee el contenido de un archivo de texto local dada su ruta.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ruta": {
                        "type": "string",
                        "description": "Ruta al archivo a leer",
                    }
                },
                "required": ["ruta"],
            },
        },
    },
]

# Diccionario para poder llamar a la funcion real a partir de su nombre
# (que es justo lo que el modelo nos va a devolver).
HERRAMIENTAS_DISPONIBLES = {
    "calculadora": calculadora,
    "fecha_actual": fecha_actual,
    "leer_archivo": leer_archivo,
}

## Paso 3: una sola llamada al modelo, con tools

Antes de armar el loop completo, veamos qué devuelve Ollama cuando le mandamos una pregunta junto con las `tools` disponibles. Si el modelo decide que necesita una herramienta, la respuesta traerá `tool_calls` en vez de (o además de) texto.

In [ ]:
mensajes = [
    {"role": "user", "content": "¿Qué día es hoy?"}
]

respuesta = ollama.chat(model=MODELO, messages=mensajes, tools=TOOLS_SCHEMA)
respuesta["message"]

Si todo salió bien, deberías ver un mensaje del modelo con `tool_calls`, algo como una petición para llamar a `fecha_actual`. El modelo **no** ejecutó la función — solo generó la intención de llamarla. Nosotros tenemos que ejecutarla y devolverle el resultado.

## Paso 4: el ciclo ReAct completo

Ahora sí, el loop del agente:

1. Mandamos el historial de mensajes + las `tools` al modelo.
2. Si pide una `tool_call` → ejecutamos la función real, agregamos el resultado al historial como un mensaje de rol `tool`, y volvemos al paso 1.
3. Si ya no pide ninguna herramienta → esa es la respuesta final, y terminamos.

Esto es exactamente el patrón `Thought → Action → Observation → ...` que vimos en la presentación, solo que aquí el "Thought" ocurre implícitamente dentro del modelo.

In [ ]:
def ejecutar_agente(pregunta: str, max_iteraciones: int = 6) -> str:
    mensajes = [
        {
            "role": "system",
            "content": (
                "Eres un asistente que puede usar herramientas para responder "
                "con precision. Usa una herramienta cuando la necesites en vez "
                "de adivinar datos como fechas o resultados de calculos."
            ),
        },
        {"role": "user", "content": pregunta},
    ]

    for paso in range(max_iteraciones):
        respuesta = ollama.chat(model=MODELO, messages=mensajes, tools=TOOLS_SCHEMA)
        mensaje = respuesta["message"]
        mensajes.append(mensaje)

        tool_calls = mensaje.get("tool_calls")
        if not tool_calls:
            # El modelo ya no quiere usar herramientas: es la respuesta final.
            return mensaje["content"]

        for llamada in tool_calls:
            nombre = llamada["function"]["name"]
            argumentos = llamada["function"]["arguments"]

            print(f"  [Action] {nombre}({argumentos})")
            funcion = HERRAMIENTAS_DISPONIBLES.get(nombre)
            if funcion is None:
                resultado = f"Error: la herramienta '{nombre}' no existe."
            else:
                resultado = funcion(**argumentos)

            print(f"  [Observation] {resultado}")
            mensajes.append({"role": "tool", "content": str(resultado), "name": nombre})

    return "No se alcanzo una respuesta final dentro del limite de iteraciones."

## Paso 5: probar el agente

Le hacemos una pregunta que necesita combinar **dos** herramientas: saber la fecha de hoy y hacer un cálculo con ese dato. Si el agente funciona bien, deberías ver la traza `Thought → Action → Observation` (aquí solo mostramos Action/Observation; el "pensamiento" queda implícito dentro del modelo) hasta llegar a la respuesta final.

In [ ]:
pregunta = "¿Qué día es hoy y cuánto es el día del mes multiplicado por 7?"
print(f"Pregunta: {pregunta}\n")

respuesta_final = ejecutar_agente(pregunta)
print(f"\nRespuesta final: {respuesta_final}")

## Paso 6: tu turno — decide la tarea

Prueba con tus propias preguntas. Algunas ideas:

- `"Lee el archivo ejemplo.txt de esta carpeta y dime de qué trata en una frase."`
- `"¿Cuánto es (25 * 4) y qué fecha es hoy?"`
- Agrega tú una cuarta herramienta (por ejemplo, contar palabras de un texto) siguiendo el mismo patrón de los pasos 1 y 2, y agrégala a `HERRAMIENTAS_DISPONIBLES` y `TOOLS_SCHEMA`.

In [ ]:
# Escribe aqui tu propia pregunta para el agente
mi_pregunta = "Lee el archivo ejemplo.txt de esta carpeta y dime de que trata en una frase."
print(ejecutar_agente(mi_pregunta))